In [ ]:
import pandas as pd
from IPython.display import display

from study.io import WORK, load, reset_directory, write_table
from study.updating import fit_score_updates, metric_rows, predict_score_updates

In [ ]:
reset_directory(WORK / 'predictions/pooled')
source_fits = load(WORK / 'models/source.pkl')

In [ ]:
# Pooled adaptation uses eICU calibration patients
rows, coefficient_rows = [], []
for repeat in source_fits['repeats']:
    predictions = pd.concat(
        [
            pd.read_parquet(
                WORK / 'predictions' / name,
                filters = [
                    ('repeat', '==', repeat),
                    ('domain', '==', 'eicu'),
                    ('source', '==', 'metavision'),
                    ('support', '==', 'shared'),
                    ('tier', 'in', ['R2', 'R3'])
                ],
            )
            for name in ('full', 'quantity')
        ],
        ignore_index = True,
    )

    # Align R2 and R3 risks by patient, restriction, and observation draw
    wide = (
        predictions.pivot(
            index = ['repeat', 'stay_id', 'hospital_id', 'partition', 'in_hospital_mortality', 'restriction', 'draw'],
            columns = 'tier',
            values = 'risk',
        )
        .reset_index()
        .rename(columns = {'in_hospital_mortality': 'label'})
    )

    for condition in ('full', 'retain_0.5', 'retain_0.25'):
        condition_rows = wide.loc[wide.restriction.eq(condition)]

        for draw, frame in condition_rows.groupby('draw'):
            # Fit score updates on reserved calibration patients and score test patients
            calibration = frame.loc[frame.partition.eq('calibration')]
            test = frame.loc[frame.partition.eq('test')]
            fitted = fit_score_updates(calibration, ridge = 1.0)
            risks = predict_score_updates(test, fitted)
            
            rows.extend(metric_rows(test, risks, repeat = repeat, restriction = condition, draw = draw))
            coefficient_rows.append(
                dict(
                    repeat = repeat,
                    restriction = condition,
                    draw = draw,
                    intercept = fitted['process_update'][0],
                    gamma = fitted['process_update'][1],
                )
            )

            pooled_parts = [
                test.drop(columns = ['R2', 'R3']).assign(risk = risk, method = method) for method, risk in risks.items()
            ]

            write_table(
                WORK / 'predictions/pooled' / f'{condition}_{draw}_{repeat}.parquet',
                pd.concat(pooled_parts, ignore_index = True),
            )

write_table(WORK / 'results/pooled_metrics.parquet', pd.DataFrame(rows))
write_table(WORK / 'results/pooled_coefficients.parquet', pd.DataFrame(coefficient_rows))

display(pd.DataFrame(rows).groupby(['restriction', 'method'], observed = True).log_loss.mean())